# Gamma-Gospel — does dealer gamma call the day? 🃏
### Is the GEX "regime read" a real dealer-hedging signal, or the VIX in a trenchcoat?

![Signal: Pre-reg](https://img.shields.io/badge/Signal-Pre--reg-8b949e?style=flat-square)
![Tradability: Pre-reg](https://img.shields.io/badge/Tradability-Pre--reg-8b949e?style=flat-square)

A viral thread says price is just the *output*; the real *input* is **dealer hedging** — long-gamma dealers fade moves (calm **range** day), short-gamma dealers chase them (violent **trend** day), and the sign of net **GEX**, knowable before the open, calls the day's character. But there's a suspect hiding in plain sight: **the VIX** — scary days are both more trending *and* put-heavy (negative-gamma). This study is **pre-registered**: no real-market results yet, only the expectation that GEX is mostly the volatility regime relabeled.

> 📓 **This is the plain-language layer.** The deep companion is **[02_for_the_quants.ipynb](02_for_the_quants.ipynb)** — same story, deeper.
>
> ⚠️ **Not investment advice.** Pre-registered design: the stamps above are expectations, to be earned (or refuted) by the run on the real SPY chain ([`examples/verify.py`](../examples/verify.py) → [`../docs/results.md`](../docs/results.md)). House style in [METHODOLOGY.md](../../../METHODOLOGY.md).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))           # study root (gamma_gospel/ lives there)
sys.path.insert(0, os.path.abspath("../../.."))      # repo root, for quantlab
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from gamma_gospel import data, signals, decompose

# Two offline tapes that share a VIX-driven confound (VIX moves BOTH the GEX sign and the day's
# character). REAL has a genuine gamma effect baked on top (beta_de=0.06); TRENCH has none.
real, t_real = data.synthetic_panel(beta_vol=0.0020, beta_de=0.060, seed=0)
trench, t_tr = data.synthetic_panel(beta_vol=0.0, beta_de=0.0, seed=0)
neg = real["neg_gamma"].astype(bool)
print(f"{len(real)} sessions | VIX on neg-gamma days {real.loc[neg,'vix'].mean():.1f} "
      f"vs pos {real.loc[~neg,'vix'].mean():.1f} | baked-in beta_de REAL={t_real.beta_de}, "
      f"TRENCH={t_tr.beta_de}")


## The answer first 🎯

| What we asked | The honest answer |
|---|---|
| Are negative-gamma days really more volatile / more trending? | ✅ **Yes, raw** — but those are also the high-VIX days. |
| Does GEX's sign add anything *over the VIX*? | ⏳ **That's the test** — the pitch lives or dies on whether the gap *survives* controlling for VIX. |
| Can our machine even tell a real effect from a relabel? | ✅ **Yes** — below it recovers a baked-in effect and unmasks a baked-in fake. |
| Would a regime *sign* be a trade? | ❌ **Not by itself** — it's a bias on the day's character, not an entry, and it needs the whole-day hold to express. |

> Desk shorthand (pre-registered): we expect **Signal `WEAK`** — most of the headline is the volatility regime relabeled — and **Tradability `MIRAGE`**. The real run confirms or breaks it.

## 1 · The claim 📣

Dealers (market-makers) don't bet on direction — they hedge. Their **gamma** says how hard they must trade per point of move. Sum it across all strikes (calls add, puts subtract, under the standard assumption that dealers are long calls and short puts) and you get net **GEX**. **GEX > 0** (long gamma): dealers fade every move → a *range* day, vol suppressed. **GEX < 0** (short gamma): dealers amplify every move → a *trend* day, vol turbocharged. The pitch says the sign — visible before the open — is the most important thing you can know about the session.

## 2 · So what? 💰

If true, you'd know each morning whether to **fade the extremes** (sell premium, trade range) or **ride breakouts** (momentum) — a regime filter worth real money, and a sign that a big chunk of intraday behaviour is *mechanically* set by dealer hedging rather than news. The subtler, likelier story: negative-gamma days really are wilder, but mostly because **negative gamma is what a high-VIX market looks like** — and you didn't need an options model to see the VIX. Confuse the two and you're paying \$7/month to be told 'it's a scary day' in greek letters.

## 3 · How we'd know 🔍

Build a daily panel: the **GEX sign** at the prior close, the next day's **character** (a range-based vol, and *directional efficiency* = |close−open|/(high−low): ~1 on a trend day, ~0 on a chop day), and the prior-close **VIX**. First check the raw gap. Then the decisive move: **partial out the VIX**. If the negative-gamma effect *survives* the control, GEX carries real information. If it *collapses*, GEX was the volatility regime in a trenchcoat. We prove the test works on the synthetic, where we baked the answer in — both versions.

In [ ]:
for name, panel, truth in [('REAL effect', real, t_real), ('TRENCHCOAT', trench, t_tr)]:
    raw = decompose.regime_gap(panel, 'de')
    p = decompose.partial_over_vix(panel, 'de')
    print(f"{name:12} raw DE gap {raw['gap']:+.3f} (t {raw['t']:+.1f})  ->  "
          f"survives VIX {p['surviving_coef']:+.3f} (t {p['surviving_t']:+.1f}), "
          f"{p['survival_share']:.0%} kept  [baked-in beta_de={truth.beta_de}]")

Notice: **both worlds show the same big raw gap** — that's the trap. Only the VIX-controlled number tells them apart. In the real-effect world ~37% survives near the baked-in 0.06; in the trenchcoat world it goes to ~0. The raw headline is *not* the evidence.

## 4 · The teardown 🔬

The whole study in one picture: the **raw** regime gap (what the pitch shows you) next to the **VIX-controlled** gap (what's actually GEX's own information), in both worlds.

In [ ]:
labels = ['raw gap', 'survives VIX-control']
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
for a, (name, panel, truth) in zip(ax, [('Real gamma effect', real, t_real),
                                        ('Trenchcoat (no effect)', trench, t_tr)]):
    raw = decompose.regime_gap(panel, 'de'); p = decompose.partial_over_vix(panel, 'de')
    vals = [raw['gap'], p['surviving_coef']]
    a.bar(labels, vals, color=['#888888', '#b22222'])
    a.axhline(0, color='k', lw=0.6)
    if truth.beta_de: a.axhline(truth.beta_de, color='#1f77b4', ls='--', lw=1.5,
                                label=f'baked-in {truth.beta_de}')
    a.set_title(name); a.legend()
    for i, v in enumerate(vals): a.text(i, v + 0.004, f'{v:+.3f}', ha='center')
ax[0].set_ylabel('negative-gamma effect on directional efficiency')
plt.suptitle('Same raw gap, opposite truth — only the VIX control separates them')
plt.tight_layout(); plt.show()

**What this means for the real tape.** On real SPY the raw gap will be large — negative gamma days *are* wilder. The verdict hinges entirely on the right-hand bar: does the negative-gamma coefficient stay up (near the blue line, real information) or fall to the floor (trenchcoat)? Given that negative gamma is essentially *defined* by a put-heavy, high-VIX book, the desk's prior — and the lesson of [Study 12](../../12-paper-prophet/) — is that the range-vol leg is almost pure VIX, and the only place GEX might keep something of its own is *directional efficiency*. [`../docs/results.md`](../docs/results.md) settles it on the real chain.

## 5 · The verdict ⚖️ *(pre-registered)*

**Signal `WEAK` (expected)** — there's a real raw gap, but we expect most of it to be the volatility regime; GEX earns `REAL` only if the VIX-controlled coefficient stays significant, especially on directional efficiency. **Tradability `MIRAGE` (expected)** — even a surviving sign is a *bias on the day's character*, not an entry, and expressing it needs a full-session hold against a round-trip spread. The synthetic proves the test is honest; the real numbers fill the stamps.

## 6 · Could you trade it? 🏦

Suppose the sign *does* survive VIX. You'd still only know *what kind* of day it is — fade extremes when long-gamma, expect trend when short-gamma. That's a **regime filter**, not a trade: to monetise it you overlay it on an actual entry, and you pay the spread either way. A directional-efficiency tilt of a few points won't pay a round-trip on SPY options or a full-day index hold by itself. The pitch even concedes this — *"you don't need to re-architect your strategy… it shifts a few things."* A \$7/month context feed can be genuinely useful and still not be an edge you can withdraw.

## 7 · Going further 🚪

- **The walls and the flip.** Does price actually respect the call-wall / put-wall as intraday extremes more than random strikes? `compute_gex` returns them; a permutation null would test it.
- **0DTE / charm / vanna.** The pitch's advanced legs — afternoon charm drift, vanna-driven rallies on IV compression. Each is its own falsifiable sub-study.
- **Intraday character.** We grade the day from daily OHLC; with intraday bars you could test the *mean-reversion vs momentum* claim minute-by-minute, not just end-of-day.
- **Run the real chain.** A reliable historical GEX needs open interest by strike, which (we checked) the free sources don't carry — Alpha Vantage's `HISTORICAL_OPTIONS` has it but is *premium*. With a paid chain key, `python examples/verify.py --fetch` accumulates the SPY panel and earns the stamps; that data reality is why this ships pre-registered.

The deep version — the HAC nested regression, the beta recovery, the trenchcoat collapse — is in [`02_for_the_quants.ipynb`](02_for_the_quants.ipynb).